10-Jan-24 03:22 AM

I could not find a .csv file online from the material from the paper </br>

Therefore I copied the table from the arxiv source code and pasted it in overleaf. I splitted the errors in their own columns by finding and replacing all /pm signs with & signs. I also replaced all &&& signs with &&&& signs to account for the error, saved as a .csv file which allowed me to open it with pandas.</br>

In this notebook I convert the calculated Fluences to Spectral energies and make a new .csv file per burst to get their energy ranges. 

In [1]:
import pandas as pd
import numpy as np
from astropy import units as u

In [2]:
def isotropic_energies(fluences, distance=362.4):
    "Convert list of fluences to spectral densities"
    "Method based on fluence.py by K.Nimmo"

    #convert from jyms to jys
    fluence_jys = np.array(fluences) * 1e-3

    dis = 1.0 * u.megaparsec
    megap_cm = dis.to(u.cm).value

    #mpc to cm
    distance_lum_cm = megap_cm*distance

    #redshift correction, see evernote for details
    #redshift is from: Ravi et al, DSA-110
    #distance is also from Ravi et al, FAST converts themselves based on PLANCK 2016
    #link: https://ui.adsabs.harvard.edu/abs/2023ApJ...949L...3R/abstract
    z = 0.0771
    red_cor = 1 / (1 + z)**2

    energy_iso = fluence_jys * 4*np.pi*(distance_lum_cm**2) * 1e-23 * red_cor
    #print(energy_iso)

    return energy_iso

In [4]:
def allen():
    '''
    Read in the .csv file, based on the delimeter
    Then I convert the Fluence to a Spectral energy per component
    '''
    allen_df = pd.read_csv('table_allen.csv', delimiter="& ", header=0, engine='python')
    allen_df.columns = allen_df.columns.str.replace(' ', '')
    
    #Converting to spectral energies and adding to the df
    spec_energies = isotropic_energies(allen_df['Fluence'].values)
    allen_df['spec-energy'] = spec_energies
    #display(allen_df)
    
    #burst id
    bursts = allen_df['burst'].unique()

    info_list = []
    
    for burst_id in bursts:
        
        allen_df_temp_burst = allen_df[(allen_df['burst'] == burst_id)]
                                       
        toa = allen_df_temp_burst['MJD'].values[0]
        fluence = np.sum(allen_df_temp_burst['Fluence'].values)
        spec_energ = np.sum(allen_df_temp_burst['spec-energy'].values)

        #Add to the list
        info_list.append([burst_id, toa, fluence, spec_energ])
    
    #Convert the fluence list to array
    info_list_arr = np.asarray(info_list)
    
    #Convert the info to dataframe
    new_data = {'id': info_list_arr[:,0], \
                'toa': info_list_arr[:,1], \
                'fluence': info_list_arr[:,2], \
                'spec-energy':info_list_arr[:,3]}
    
    df_new_fluence = pd.DataFrame(data=new_data)
    #display(df_new_fluence)
    
    print(f"range of spectral energies: {df_new_fluence['spec-energy'].min()} - {df_new_fluence['spec-energy'].max()}")
    
    name_csv = "R117_allen_omar.csv"
    df_new_fluence.to_csv(name_csv, header=True, index=True, na_rep='NA')
    
    display(df_new_fluence)
    return

allen()

range of spectral energies: 1.1486061384967963e+30 - 1.5283098635024644e+32


,id,toa,fluence,spec-energy
0,1.0,59872.442330,21.88,2.963621e+30
1,2.0,59873.350499,215.34,2.916755e+31
2,3.0,59875.267211,26.87,3.639510e+30
3,4.0,59880.272023,634.16,8.589623e+31
4,5.0,59880.395558,167.72,2.271748e+31
5,6.0,59880.402931,223.48,3.027011e+31
6,7.0,59882.270717,24.15,3.271089e+30
7,8.0,59882.472648,16.52,2.237615e+30
8,9.0,59883.109767,52.06,7.051466e+30
9,10.0,59883.450239,91.89,1.244639e+31
